# L51 Proof Replay

This notebook replays the bounded Engineering contract-pack proof on fixed repo state.

## Proof Focus

- One declared Engineering authority
- Seven contract-only Engineering personas
- One deterministic zero-action claim
- One deterministic blocked-state rule


## Fixed-State Replay

Re-run the authority and runtime checks against the same repo state.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import yaml

from smarthaus_common.department_pack import build_department_pack
from smarthaus_common.json_store import JsonStore

repo_root = Path.cwd()
authority_path = repo_root / 'registry' / 'department_pack_engineering_v1.yaml'
payload = yaml.safe_load(authority_path.read_text(encoding='utf-8'))

assert payload['department']['id'] == 'engineering'
assert payload['kpis']['required_personas'] == 7
assert payload['kpis']['required_active_personas'] == 0
assert payload['kpis']['supported_action_count'] == 0
assert payload['personas']['ai-engineer']['coverage_status'] == 'persona-contract-only'


In [ ]:
with TemporaryDirectory() as temp_dir:
    pack = build_department_pack('engineering', store=JsonStore(Path(temp_dir)))

assert pack['department']['id'] == 'engineering'
assert pack['summary']['persona_count'] == 7
assert pack['summary']['active_persona_count'] == 0
assert pack['summary']['registry_backed_persona_count'] == 0
assert pack['summary']['supported_action_count'] == 0
assert pack['summary']['pack_state'] == 'blocked'
assert {persona['coverage_status'] for persona in pack['personas']} == {'persona-contract-only'}

assert pack['bounded_claims']['supported'][2] == 'deterministic blocked-state projection while engineering remains contract-only'
